# Phase 4 — Kaggle X3D tensor cache and baseline training

Select a Kaggle **T4 x2 GPU** accelerator and enable Internet for the first pretrained-weight download. The notebook caches RWF-2000 as resumable RGB uint8 `.npy` memory maps, verifies the files, launches distributed mixed-precision X3D-S training across both GPUs, and evaluates the held-out RWF test split only after training finishes. Decoding is capped at two CPU workers; training uses two DataLoader workers per GPU process. Add your Kaggle `WANDB_API_KEY` secret to enable online tracking; training still runs if the logger falls back to TensorBoard/no-op.


## 1. Clone/update the code and install video-model dependencies

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception as exc:
    print('W&B Kaggle secret unavailable; logger will use its fallback:', exc)

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # For a report run, pin this to a commit SHA.
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
gpu_count = torch.cuda.device_count()
print('PyTorch:', torch.__version__, '| visible GPUs:', gpu_count)
for i in range(gpu_count):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
if gpu_count < 1:
    raise RuntimeError('Select the Kaggle T4 x2 GPU accelerator before continuing.')
if gpu_count != 2:
    print('This session does not expose two GPUs; the runner will use all visible GPUs.')

## 2. Locate and link the attached RWF-2000 dataset

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
rwf_roots = sorted({p.parent.parent for p in INPUT_ROOT.rglob('Fight') if p.is_dir() and p.parent.name == 'train' and (p.parent.parent / 'val' / 'Fight').is_dir() and (p.parent.parent / 'train' / 'NonFight').is_dir()})
print("RWF roots found:", *rwf_roots, sep="\n  ")
if len(rwf_roots) != 1:
    raise RuntimeError('Attach exactly one RWF-2000 input containing train/Fight, train/NonFight, val/Fight, and val/NonFight.')
RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
target = RAW_DIR / 'rwf2000'
if target.is_symlink():
    target.unlink()
elif target.exists():
    if any(target.iterdir()):
        raise FileExistsError(f'Expected an empty or symlink path at {target}; inspect it before replacing.')
    target.rmdir()
target.symlink_to(rwf_roots[0], target_is_directory=True)
print(f'Linked {target} -> {rwf_roots[0]}')

## 3. Build/resume the full-frame uint8 tensor cache

The cache uses 32 evenly spaced candidate frames per clip at 176×176 RGB, matching the prior X3D notebook's `round(160 × 1.1)` cache size. Training selects one of two candidates in each of 16 temporal segments, then resizes to the 160×160 model input on the GPU. It writes a memory-mapped tensor per split and resumes incomplete rows after interruption. The expected full-frame tensor data is about 5.5 GiB, before labels, indices, and metadata.


In [ ]:
TENSOR_CACHE = Path('/kaggle/working/tensor_cache/rwf2000')
subprocess.run([sys.executable, 'scripts/cache_tensors.py', '--manifest', 'data/splits/rwf2000.csv', '--config', 'configs/preprocess.yaml', '--model', 'x3d_s', '--cache-dir', str(TENSOR_CACHE), '--mode', 'full', '--workers', '2'], check=True)

## 4. Verify every split, tensor shape, dtype, and row checksum

In [ ]:
subprocess.run([sys.executable, 'scripts/verify_tensor_cache.py', '--cache-dir', str(TENSOR_CACHE), '--input-tag', 'rwf_full_f16_s160_v1', '--manifest', 'data/splits/rwf2000.csv', '--report-dir', '/kaggle/working/tensor_cache/reports'], check=True)
print('Verification report:', Path('/kaggle/working/tensor_cache/reports/verification_report.md').read_text())

## 5. Train X3D-S with both T4 GPUs

The training script uses DistributedDataParallel, one process per visible GPU, FP16 autocast/gradient scaling, channels-last-3D tensors, and a bounded CPU loader. Checkpoints, train/validation curves, and the final train/validation gap are saved under `/kaggle/working/experiments/x3d_s_rwf2000`. The held-out test split is not used for model selection.


In [ ]:
GPU_COUNT = torch.cuda.device_count()
TRAIN_OUTPUT = Path('/kaggle/working/experiments/x3d_s_rwf2000')
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)
torchrun_command = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nnodes=1',
    f'--nproc_per_node={GPU_COUNT}', 'src/violence_detection/train.py',
    '--config', 'configs/train_x3d.yaml', '--cache-dir', str(TENSOR_CACHE),
    '--output-dir', str(TRAIN_OUTPUT), '--mode', 'full'
]
print("Launching:", torchrun_command)
subprocess.run(torchrun_command, check=True)

## 6. Evaluate the selected checkpoint on the official RWF validation-as-test split

In [ ]:
TEST_OUTPUT = TRAIN_OUTPUT / 'test'
subprocess.run([
    sys.executable, 'src/violence_detection/evaluate.py',
    '--checkpoint', str(TRAIN_OUTPUT / 'best.pt'),
    '--config', 'configs/train_x3d.yaml', '--cache-dir', str(TENSOR_CACHE),
    '--output-dir', str(TEST_OUTPUT), '--split', 'test'
], check=True)
print('Test results:', (TEST_OUTPUT / 'test_results.md').read_text())

## 7. Inspect artifacts for Kaggle output download

In [ ]:
for root in [Path('/kaggle/working/tensor_cache/reports'), TRAIN_OUTPUT]:
    print(f"\n{root}")
    for path in sorted(root.rglob("*")):
        if path.is_file():
            print(path, path.stat().st_size, "bytes")